# Blob to Bronze — Transaction Ingestion

Incrementally ingests CSV files arriving at the blob source volume into the bronze Delta table `retail_q.blob_bronze.transactions` using Auto Loader (`cloudFiles`).

In [0]:
# Configuration
source_path = "/Volumes/retail_q/volumes/blob_source/transaction_source/"
target_table = "retail_q.blob_bronze.transactions"

# Locations for schema inference and streaming checkpoints
schema_location = "/Volumes/retail_q/volumes/blob_source/_schemas/transactions"
checkpoint_location = "/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions"

In [0]:
# Auto Loader: incrementally read CSV files from the blob source
raw_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

# Add metadata columns for traceability
from pyspark.sql.functions import current_timestamp, col

bronze_df = raw_df.withColumn("_ingested_at", current_timestamp()).withColumn("_source_file", col("_metadata.file_path"))


In [0]:
# Write the streaming DataFrame to the bronze Delta table
(bronze_df.writeStream
   .format("delta")
   .option("checkpointLocation", checkpoint_location)
   .option("mergeSchema", "true")
   .trigger(availableNow=True)  # process all available files then stop; change to processingTime='1m' for continuous
   .toTable(target_table))

In [0]:
%sql
SELECT * FROM retail_q.blob_bronze.transactions;

In [0]:
%sql
SELECT COUNT(*) Total_Transactions FROM retail_q.blob_bronze.transactions;

In [0]:
%sql
DROP table retail_q.retail_silver.product_catalog;

In [0]:
%sql
SELECT *
FROM retail_q.salesforce_bronze.opportunity
where StageName NOT IN ('Prospecting','Closed Won','Closed Lost');

In [0]:
%sql
-- DQ Rule: 'valid stage' — StageName must be one of ('Prospecting','Closed Won','Closed Lost')
-- Returns the count of rows that VIOLATE the rule (invalid) and the count of rows that PASS (valid)
SELECT
  SUM(CASE WHEN StageName NOT IN ('Prospecting','Closed Won','Closed Lost') OR StageName IS NULL THEN 1 ELSE 0 END) AS invalid_stage_count,
  SUM(CASE WHEN StageName IN ('Prospecting','Closed Won','Closed Lost') THEN 1 ELSE 0 END) AS valid_stage_count
FROM retail_q.salesforce_bronze.opportunity;

In [0]:
%sql
select * from retail_q.salesforce_bronze.opportunity;

In [0]:
%sql
DROP TABLE retail_q.retail_silver.product_catalog;
DROP TABLE retail_q.retail_silver.product_catalog_cdc;